In [ ]:
import pandas as pd
import requests
import os
import time
from tqdm import tqdm
import concurrent.futures
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

API_BASE_URL = "https://massbank.eu/MassBank-api/records"
INPUT_FILE = "data.xlsx"
OUTPUT_FILE = "ms.csv"
MAX_WORKERS = 5
RETRY_COUNT = 3
BATCH_SIZE = 100

def get_processed_inchikeys(output_file):
    if not os.path.exists(output_file):
        return set()
    try:
        df = pd.read_csv(output_file, on_bad_lines='warn')
        if 'InChIKey' in df.columns:
            return set(df['InChIKey'].dropna().unique())
        else:
            return set()
    except (pd.errors.EmptyDataError, FileNotFoundError):
        return set()

def create_session_with_retries(retries=RETRY_COUNT):
    session = requests.Session()
    retry_strategy = Retry(
        total=retries,
        backoff_factor=0.3,
        status_forcelist=[500, 502, 503, 504],
    )
    adapter = HTTPAdapter(max_retries=retry_strategy)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

def fetch_records_by_inchikey(inchi_key, session):
    if pd.isna(inchi_key) or inchi_key.strip() == '':
        tqdm.write(f"Warning: InChIKey is empty, skipping query.")
        return None

    url = f"{API_BASE_URL}?inchi_key={inchi_key}"
    try:
        response = session.get(url, timeout=30)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        tqdm.write(f"Warning: Network error for InChIKey '{inchi_key}': {e}")
        return None
    except ValueError:
        tqdm.write(f"Warning: Cannot parse JSON for InChIKey '{inchi_key}'. No records found.")
        return None

def extract_nested_value(data, key_path, default='N/A'):
    try:
        for key in key_path:
            data = data[key]
        return data
    except (KeyError, TypeError, IndexError):
        return default

def process_row(original_row_tuple, session):
    index, original_row = original_row_tuple
    inchi_key = original_row['InChIKey']

    records = fetch_records_by_inchikey(inchi_key, session)

    if not records:
        return pd.DataFrame([{'InChIKey': inchi_key, 'Peaks': 'NOT_FOUND'}])

    all_results_for_row = []
    for record in records:
        peak_data = extract_nested_value(record, ['peak', 'peak', 'values'], [])

        peak_str = 'NO_PEAK_DATA'
        if peak_data and isinstance(peak_data, list):
            peak_pairs = []
            for peak in peak_data:
                mz = peak.get('mz')
                rel_int = peak.get('rel')
                if mz is not None and rel_int is not None:
                    peak_pairs.append(f"{mz}:{rel_int}")

            if peak_pairs:
                peak_str = ' '.join(peak_pairs)

        result_dict = {'InChIKey': inchi_key, 'Peaks': peak_str}
        all_results_for_row.append(result_dict)

    if not all_results_for_row:
        return pd.DataFrame([{'InChIKey': inchi_key, 'Peaks': 'NO_RECORDS_PROCESSED'}])

    return pd.DataFrame(all_results_for_row)

def save_batch_to_csv(batch, filename):
    if not batch: return
    final_df_batch = pd.concat(batch, ignore_index=True)
    header = not os.path.exists(filename)
    final_df_batch.to_csv(filename, mode='a', header=header, index=False)

def main():
    try:
        input_df = pd.read_excel(INPUT_FILE)
    except FileNotFoundError:
        print(f"Error: Input file '{INPUT_FILE}' not found.")
        return

    processed_keys = get_processed_inchikeys(OUTPUT_FILE)
    df_to_process = input_df[~input_df['InChIKey'].isin(processed_keys)].copy()

    if df_to_process.empty:
        print(f"All InChIKeys from '{INPUT_FILE}' have been processed.")
        return

    print(f"Found {len(input_df)} rows total, {len(df_to_process)} rows to process.")

    results_batch = []
    session = create_session_with_retries()

    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_row = {executor.submit(process_row, row, session): row for row in df_to_process.iterrows()}

        for future in tqdm(concurrent.futures.as_completed(future_to_row), total=len(df_to_process), desc="Processing rows"):
            try:
                result_df = future.result()
                if result_df is not None:
                    results_batch.append(result_df)

                if len(results_batch) >= BATCH_SIZE:
                    save_batch_to_csv(results_batch, OUTPUT_FILE)
                    results_batch = []

            except Exception as exc:
                tqdm.write(f"Exception occurred while processing a row: {exc}")

    save_batch_to_csv(results_batch, OUTPUT_FILE)

    print(f"\nProcessing complete! Results saved to '{OUTPUT_FILE}'.")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

data = pd.read_csv('data.csv')

X = data.drop(['label'], axis=1)
y = data['label']

unique_names = X['name'].unique()

train_names, test_names = train_test_split(
    unique_names, 
    test_size=0.2, 
    random_state=42
)

train_mask = X['name'].isin(train_names)
test_mask = X['name'].isin(test_names)

X_train_raw = X[train_mask].copy()
X_test_raw = X[test_mask].copy()
y_train = y[train_mask].copy()
y_test = y[test_mask].copy()

use_smiles = False
inchikey_to_smiles = {}

if use_smiles:
    X_train_raw['SMILES'] = X_train_raw['name'].map(inchikey_to_smiles)
    X_test_raw['SMILES'] = X_test_raw['name'].map(inchikey_to_smiles)

train_names_saved = X_train_raw['name'].copy()
test_names_saved = X_test_raw['name'].copy()

if use_smiles:
    train_smiles_saved = X_train_raw['SMILES'].copy()
    test_smiles_saved = X_test_raw['SMILES'].copy()
    X_train = X_train_raw.drop(columns=['name', 'SMILES'])
    X_test = X_test_raw.drop(columns=['name', 'SMILES'])
else:
    X_train = X_train_raw.drop(columns=['name'])
    X_test = X_test_raw.drop(columns=['name'])

continuous_features = X_train.select_dtypes(include=['float64', 'int64']).columns

X_train = X_train[continuous_features]
X_test = X_test[continuous_features]

for column in X_train.columns:
    if X_train[column].dtype in ['float64', 'int64']:
        median_value = X_train[column].median() if X_train[column].notna().any() else 0
        X_train[column].fillna(median_value, inplace=True)
        if column in X_test.columns:
            X_test[column].fillna(median_value, inplace=True)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_resampled)
X_test_scaled = scaler.transform(X_test)

X_train = pd.DataFrame(
    X_train_scaled, 
    columns=continuous_features
)

X_test = pd.DataFrame(X_test_scaled, columns=continuous_features, index=X_test.index)

y_train = y_train_resampled